In [1]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
import pyarrow.parquet as pq
import sentencepiece as spm
import math
import os
torch.manual_seed(42)

In [2]:
TOKENIZER_DIR = "models/my_tokenizer.model"
CHECKPOINT_DIR = "models/checkpoints/"
config = mn.Config()
config.n_embd = 256
config.n_head = 8
config.n_layer = 8
config.mini_batch_size = 16 # B
config.context_length = 256 # T
config.total_batch_size = 1 * config.mini_batch_size * config.context_length # MUST BE DIVISBLE by (B*T)
config.device = 'cuda'
config.vocab_size = 32768

max_iter = 20000
B = config.mini_batch_size
T = config.context_length
grad_accum_steps = config.total_batch_size // (B*T)
assert config.total_batch_size % (B * T) == 0, "Total batch size must be divisble by (B*T)"

In [4]:
import remotedataset as rd
import sentencepiece as spm

if not os.path.exists(TOKENIZER_DIR):
    hf_dataset = rd.get_interleaved_dataset()
    rd.train_tokenizer(rd.create_tokenizer_corpus(hf_dataset)) 
    
sp = spm.SentencePieceProcessor(model_file = TOKENIZER_DIR) # pyright: ignore[reportCallIssue]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/234 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/184 [00:00<?, ?it/s]

In [ ]:
hf_dataset = rd.get_interleaved_dataset()
val_hf = hf_dataset.take(1000)
train_hf = hf_dataset.skip(1000)

train_ds = rd.PackedDataset(train_hf, sp, config.mini_batch_size, config.context_length)
val_ds = rd.PackedDataset(val_hf, sp, config.mini_batch_size, config.context_length)
train_dataloader = torch.utils.data.DataLoader(train_ds, batch_size=None, num_workers=2, pin_memory=True, prefetch_factor=4, persistent_workers=True)
val_dataloader = torch.utils.data.DataLoader(val_ds, batch_size=None, num_workers=0, pin_memory=True)#, prefetch_factor=4, persistent_workers=True)
def infinite_dataloader(dataloader):
    while True:
        for batch in dataloader:
            return batch
train_dataloader = iter(infinite_dataloader(train_dataloader))

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/234 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/184 [00:00<?, ?it/s]

In [5]:
print(sp.Encode("Lily found a needle.")) # pyright: ignore[reportAttributeAccessIssue]
print(sp.Decode(sp.Encode("Lily found a needle.")))

[7006, 1189, 267, 12363, 32692]
Lily found a needle.


In [7]:
def get_lr(step):
    max_lr = 1e-3
    min_lr = 1e-4
    warmup_steps=200
    max_steps=20000
    if step < warmup_steps:
        return max_lr * (step+1)/warmup_steps 
    if step > max_steps:
        return min_lr
    decay_ratio = (step-warmup_steps)/(max_steps-warmup_steps)
    coefficient = 0.5 * (1.0 + math.cos(decay_ratio * math.pi)) # starts at 1 reduces to 0
    return min_lr + coefficient * (max_lr - min_lr)
    

In [8]:
torch.set_float32_matmul_precision('high')
model = mn.Transformer(config, dtype=torch.float32, device=config.device)
params = model.parameters()

In [9]:
optimizer_adamw = model.configure_optimizer(weight_decay=0.1, learning_rate=get_lr(0), device=config.device)

In [ ]:
print(sum(p.numel() for p in params)/1e6, "M params")

In [9]:
# # Load the list of tensors
# loaded_params = torch.load("models/params_new.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

In [30]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_step_400.pt")
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint at {checkpoint_path}, loading...")
    checkpoint = torch.load(checkpoint_path, map_location=config.device, weights_only=False)
    
    model.load_state_dict(checkpoint['model'])
    optimizer_adamw.load_state_dict(checkpoint['optimizer'])
    step = checkpoint['step']
    last_val_loss = checkpoint['val_loss']
    print(f"Checkpoint loaded. Continuing from step {step}")
else:
    step = 0
    last_val_loss = 0.0

Found checkpoint at models/checkpoints/checkpoint_step_400.pt, loading...
Checkpoint loaded. Continuing from step 400


In [31]:
model_optim = torch.compile(model)

In [ ]:
def get_val_loss():
    max_batches = 10
    val_loss = []
    with torch.no_grad():
        for i, (xs, ys) in enumerate(val_dataloader):      
            if i >= max_batches:
                break
            xs = xs.to(device=config.device, non_blocking=True)
            ys= ys.to(device=config.device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                _, loss = model_optim(xs, targets=ys)
            val_loss.append(loss.item())
    return sum(val_loss)/len(val_loss)

In [ ]:
data_iter = iter(train_dataloader)
lossi = []
timei = []
mean_stat_batches = 1
print_batches = 1
val_batches = 50
checkpoint_batches = 50
permanent_checkpoint_batches = 100

scalar = torch.amp.GradScaler('cuda') # pyright: ignore[reportPrivateImportUsage]

for _ in range(step, max_iter):
    t_start = time.time()
    optimizer_adamw.zero_grad(set_to_none=True)
    
    # forward pass
    loss_accum = torch.tensor(0.0, device=config.device)
    for micro_step in range(grad_accum_steps):
        xs, ys= next(data_iter) # loop_around=True so we don't need to worry about StopIteration exception
        xs = xs.to(device=config.device, non_blocking=True)
        ys= ys.to(device=config.device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits, loss = model_optim(xs, targets=ys)
        loss = loss / grad_accum_steps # normalizer to avoid loss gradients being exagerrated by grad_accum_steps times.
        loss_accum += loss.detach()
        # backward pass
        scalar.scale(loss).backward()
    
    scalar.unscale_(optimizer_adamw)
    norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    
    # update
    lr = get_lr(step)
    for param_group in optimizer_adamw.param_groups:
        param_group['lr'] = lr
        
    scalar.step(optimizer_adamw)
    
    scalar.update()
    
    torch.cuda.synchronize()
    t_end = time.time()
    
    # track stats    
    lossi.append(loss_accum.item())
    timei.append(t_end-t_start)
    if step % print_batches == 0:
        lossi = lossi[-mean_stat_batches:]
        timei = timei[-mean_stat_batches:]
        train_loss = sum(lossi)/len(lossi)
        if step % val_batches == 0:
            val_loss = get_val_loss()
            last_val_loss = val_loss
        else: 
            val_loss = 0.0
        dt = sum(timei)/len(timei)
        tokens_per_sec = (xs.shape[0] * xs.shape[1] * grad_accum_steps)/dt # (B*T)/dt
        print(f"Step {step}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | dt: {dt*1000:.2f}ms | {int(tokens_per_sec):,d} tok/s")
    
    step += 1
    
    # checkpointing, after current step is OVER. hence next batch should continue fron NEXT STEP.
    if step > 0 and (step-1) % checkpoint_batches == 0:
        checkpoint = {
            'model': model.state_dict(),
            'optimizer': optimizer_adamw.state_dict(),
            'config': config,
            'step': step,
            'val_loss': last_val_loss
        }
        
        torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, "checkpoint_latest.pt"))
        print(f"Checkpoint saved for step {step}.")
        if (step-1) % permanent_checkpoint_batches == 0:
            torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, f"checkpoint_step_{step}.pt"))
            


Step 400: Train Loss: 6.8777 | Val Loss: 6.8430 | dt: 5653.98ms | 724 tok/s
Checkpoint saved for step 400.
Step 401: Train Loss: 6.5103 | Val Loss: 0.0000 | dt: 101.64ms | 40,297 tok/s
Step 402: Train Loss: 6.8965 | Val Loss: 0.0000 | dt: 96.56ms | 42,418 tok/s
Step 403: Train Loss: 6.5184 | Val Loss: 0.0000 | dt: 104.02ms | 39,377 tok/s
Step 404: Train Loss: 6.3955 | Val Loss: 0.0000 | dt: 94.13ms | 43,515 tok/s
Step 405: Train Loss: 6.9170 | Val Loss: 0.0000 | dt: 99.53ms | 41,154 tok/s
Step 406: Train Loss: 6.6480 | Val Loss: 0.0000 | dt: 99.99ms | 40,963 tok/s
Step 407: Train Loss: 6.8649 | Val Loss: 0.0000 | dt: 102.06ms | 40,135 tok/s
Step 408: Train Loss: 6.4149 | Val Loss: 0.0000 | dt: 96.52ms | 42,437 tok/s
Step 409: Train Loss: 6.7629 | Val Loss: 0.0000 | dt: 93.77ms | 43,681 tok/s
Step 410: Train Loss: 6.5360 | Val Loss: 0.0000 | dt: 97.44ms | 42,036 tok/s
Step 411: Train Loss: 5.8149 | Val Loss: 0.0000 | dt: 94.41ms | 43,383 tok/s
Step 412: Train Loss: 6.6864 | Val Loss: 0.

KeyboardInterrupt: 

In [17]:
val_loss = get_val_loss()
val_loss

: 

In [ ]:
SAVE = True
if SAVE:
    params_list = list(model.parameters())
    torch.save(params_list, "models/kaggle_test.pth")

In [ ]:
print(mn.prompt("Tim loves to eat toast. So he asked his mom to cook some", model=model, tokenizer=sp, new_tokens=256, debug=True, device=config.device))

</s>
 Tim loves to eat toast. So he asked his mom to cook some more jam on the stove. His mom said, "You should read cows to be good, Tim." Tim quietly put on his toast, and sets the chair behind his back. He pebbles on the big bowl and pretends it is fuel for the itch. With his mom's help, they scrubber the bowl and put it back in the bowl. It was very hot now! Tim closed his eyes and tried to keep the inside, but it was stuck. He asked his mom, "Mom, why is such the m sandwiches falling from the consequences?" His mom smiled and said, "Sometimes the other food that it is stuck outside on the stove, but with shopping things." Tim nodded. He knew that he would be humble just like struggling to find the food rain him when he read.</s>
 Once upon a time there were two friends, Jack and Fred. They were playing in the garden. They were exploring when suddenly Bob heard that they were confused. He said to Fred, "Coming Prunes!" Fred replied, "Watch me, I'm going to please!" Fred smiled and 